In [104]:
import pandas as pd
from transformers import pipeline
from transformers import RobertaModel, RobertaTokenizer, RobertaForSequenceClassification
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


In [100]:
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")

model = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

candidate_labels = ["environment", "unions", "gun-violence", "healthcare"]


In [110]:
notes = pd.read_csv("test_notes.csv")

In [111]:
import pandas as pd
from transformers import pipeline

def get_labels(notes_with_ids, model, candidate_labels):
    results = []

    classifier = pipeline("zero-shot-classification", model=model)

    for index, row in notes_with_ids.iterrows():
        event_id = row['event_id']
        note = row['note']  
        sequence_to_classify = note
        output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
        scores_for_note = {label: score for label, score in zip(output["labels"], output["scores"])}
        scores_for_note['Event ID'] = event_id
        results.append(scores_for_note)

    df = pd.DataFrame(results)
    df.set_index('Event ID', inplace=True)
    df = df.T

    return df




In [112]:
results = get_labels(notes, model, candidate_labels)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [113]:
results = results.T

In [114]:
result = results.apply(lambda row: row == row.max(), axis=1).astype(int)
